In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from scipy import stats
from collections import defaultdict

# 1. Вероятность больших перекосов
Допустим у нас есть по 1000 объектов из двух страт. Будем сэмплировать выборки разного размера, посмотрим как сильно они будут несбалансированы.

In [ ]:
N = 1000
n_iter = 10000

data = np.zeros(2 * N)
data[:N] = 1

res = []
sample_sizes = [10, 20, 50, 100, 500]
for sample_size in sample_sizes:
    list_part_second_strata = []
    for _ in range(n_iter):
        sample = np.random.choice(data, sample_size, False)
        list_part_second_strata.append(np.mean(sample))
    res.append(list_part_second_strata)

df_res = pd.DataFrame(res, index=sample_sizes).T

In [ ]:
sns.violinplot(data=df_res, bw=.1, cut=1, linewidth=1)
plt.title('Распределение доли одной из страт в зависимости от размера выборки')
plt.grid()
plt.show()

При увеличении размера выборки вероятность того, что выборка будет сильно несбалансированная уменьшается.

# 2. Дисперсия при стратификации
Сравним как меняется дисперсия при стратификации и случайном сэмплировании для различных исходных данных.

In [ ]:
def calc_var_mean_strat(
    strata_first: np.array, strata_second: np.array,
    sample_size=100, n_iter=1000
):
    """Считаем дисперсию стратифицированного среднего при стратифицированном сэмплировании.

    strata_first - множества значений первой страты
    strata_second - множества значений второй страты
    sample_size - размер сэмплируемой выборки
    n_iter - кол-во итераций сэмплирования

    return: среднее средних, дисперсия средних, список средних
    """
    len_first = len(strata_first)
    len_second = len(strata_second)

    weight_first_strata = len_first / (len_first + len_second)
    weight_second_strata = len_second / (len_first + len_second)
    
    sample_size_first = int(round(sample_size * weight_first_strata))
    sample_size_second = sample_size - sample_size_first
    
    means_list = []
    for _ in range(n_iter):
        sample_first = np.random.choice(strata_first, sample_size_first, False)
        sample_second = np.random.choice(strata_second, sample_size_second, False)
        means_list.append(
            weight_first_strata * np.mean(sample_first) + weight_second_strata * np.mean(sample_second)
        )
    return np.mean(means_list), np.var(means_list), means_list


def calc_var_mean_srs_strat(
    strata_first: np.array, strata_second: np.array,
    sample_size=100, n_iter=1000
):
    """Считаем дисперсию среднего стратифицированного при случайном сэмплировании.
    
    strata_first - множества значений первой страты
    strata_second - множества значений второй страты
    sample_size - размер сэмплируемой выборки
    n_iter - кол-во итераций сэмплирования
    
    return: среднее средних, дисперсия средних, список средних
    """
    len_first = len(strata_first)
    len_second = len(strata_second)
    len_data = len_first + len_second
    data = np.concatenate((strata_first, strata_second))
    
    weight_first_strata = len_first / (len_first + len_second)
    weight_second_strata = len_second / (len_first + len_second)
    
    means_list = []
    for _ in range(n_iter):
        sample_indexes = np.random.choice(np.arange(len_data), sample_size, False)
        sample_indexes_first_strata = sample_indexes[sample_indexes < len_first]
        sample_indexes_second_strata = sample_indexes[sample_indexes >= len_first]

        if (len(sample_indexes_first_strata) == 0) or (len(sample_indexes_second_strata) == 0):
            mean_strat = np.mean(data[sample_indexes])
        else:
            mean_first = np.mean(data[sample_indexes_first_strata])
            mean_second = np.mean(data[sample_indexes_second_strata])
            mean_strat = weight_first_strata * mean_first + weight_second_strata * mean_second
        means_list.append(mean_strat)
    return np.mean(means_list), np.var(means_list), means_list


def calc_var_mean_srs(
    strata_first: np.array, strata_second: np.array,
    sample_size=100, n_iter=1000
):
    """Считаем дисперсию обычного среднего при случайном сэмплировании.
    
    strata_first - множества значений первой страты
    strata_second - множества значений второй страты
    sample_size - размер сэмплируемой выборки
    n_iter - кол-во итераций сэмплирования
    
    return: среднее средних, дисперсия средних, список средних
    """
    data = np.concatenate((strata_first, strata_second))
    
    means_list = []
    for _ in range(n_iter):
        sample_data = np.random.choice(data, sample_size, False)
        means_list.append(np.mean(sample_data))
    return np.mean(means_list), np.var(means_list), means_list


def run_calc_vars(
    strata_first: np.array, strata_second: np.array,
    sample_size=100, n_iter=1000,
    show=False
):
    """Cчитает средние и дисперсии средних значений, посчитанных при
    сэмплировании разными способами.
    """
    function_dict = {
        'strat': calc_var_mean_strat,
        'srs_strat': calc_var_mean_srs_strat,
        'srs': calc_var_mean_srs
    }
    res_dict = {}
    for function_name, function in function_dict.items():
        mean_, var_, _ = function(strata_first, strata_second, sample_size, n_iter)
        res_dict[f'mean {function_name}'] = mean_
        res_dict[f'var {function_name}'] = var_
        if show:
            print(f'{function_name:<10} mean {mean_:0.4f}, var {var_:0.4f}')
    return res_dict

#### посмотрим, что функции адекватно работают
При одинаковых распределениях в стратах дисперсии должны быть одинаковы

In [ ]:
sample_size = 100
n_iter = 10000
data_first = np.random.normal(0, 1, 100)
data_second = np.random.normal(0, 1, 100)
_ = run_calc_vars(data_first, data_second, sample_size, n_iter, show=True)

Если сделаем различные дисперсии выборок, то дисперсии средних для разных методов должны увеличиться и остаться одинаковыми, тк стратификация уменьшает дисперсию за счёт различия средних значений между стратами

In [ ]:
data_first = np.random.normal(0, 1, 100)
data_second = np.random.normal(0, 2, 100)
_ = run_calc_vars(data_first, data_second, sample_size, n_iter, show=True)

Теперь сделаем различные средний значения у страт, дисперсии у стратифицированных подходов должны стать меньше чем у случайного сэмплирования

In [ ]:
data_first = np.random.normal(0, 1, 100)
data_second = np.random.normal(1, 1, 100)
_ = run_calc_vars(data_first, data_second, sample_size, n_iter, show=True)

### Построим зависимость дисперсии оценок среднего в зависимости от размера выборки

In [ ]:
def plot_mean_var(df):
    """Рисует графики средних и дисперсий по значениям в столбцах датафрейма."""
    columns = df.columns
    columns_var = [c for c in columns if 'var' in c]
    columns_mean = [c for c in columns if 'mean' in c]
    
    fig = plt.figure(figsize=[12, 4])
    
    ax_one = plt.subplot(121)
    df[columns_var].plot(ax=ax_one)
    plt.grid()

    ax_two = plt.subplot(122)
    df[columns_mean].plot(ax=ax_two)
    
    ax_two.set_ylim([0, df[columns_mean].values.max() * 1.1])
    plt.grid()

In [ ]:
data_first = np.random.normal(0, 1, 100)
data_second = np.random.normal(2, 1, 100)
sample_sizes = np.arange(5, 100, 5)
n_iter = 10000
res = []
for sample_size in tqdm(sample_sizes):
    res.append(run_calc_vars(data_first, data_second, sample_size, n_iter, show=False))

df = pd.DataFrame(res, index=sample_sizes)

In [ ]:
plot_mean_var(df)

Численные эксперимент согласуется с теорией. Дисперсия располагаются так
$$var_{srs} \geq var_{srs\_strat} \geq var_{strat}$$

При увеличении размера выборки $var_{srs\_strat} \approx var_{strat}$.

### Построим зависимость дисперсий от величины отличия средних между стратами

In [ ]:
np.random.seed(45)

sample_size = 20
n_iter = 10000
mean_deltas = np.linspace(0, 5, 21)

res = []
for mean_delta in tqdm(mean_deltas):
    data_first = np.random.normal(0, 1, 100)
    data_second = np.random.normal(0 + mean_delta, 1, 100)
    res.append(run_calc_vars(data_first, data_second, sample_size, n_iter, show=False))

df = pd.DataFrame(res, index=mean_deltas)

In [ ]:
plot_mean_var(df)

При увеличении отличий между стратами дисперсии strat и srs_strat практически не изменяются, а дисперсия srs растёт.

# 3. Пример оценки аб теста и проверка корректности
Сгенерируем данные и проверим корректность методов

In [ ]:
def plot_pvalue_distribution_power(dict_pvalues, alpha=0.05):
    """Рисует графики распределения pvalue."""
    X = np.linspace(0, 1, 1000)
    for key, pvalues in dict_pvalues.items():
        Y = [np.mean(pvalues <= x) for x in X]
        prob_p = np.mean(np.array(pvalues) < alpha)
        plt.plot(X, Y, label=f'{key}, prob_p={prob_p:0.2f}')
    plt.plot([alpha, alpha], [0, 1], '--k', alpha=0.8)
    plt.plot([0, 1], [0, 1], '--k', alpha=0.8)
    plt.title('Оценка распределения p-value', size=16)
    plt.xlabel('p-value', size=12)
    plt.legend(fontsize=12)
    plt.grid()
    plt.show()
    

def calc_strat_mean(df, weights):
    """Считает стратифицированное среднее.

    :param df (pd.DataFrame): датафрейм с целевой метрикой и данными для стратификации
    :param weights (pd.Series): маппинг {название страты: вес страты в популяции}
    
    :return strat_mean (float): стратифицированное среднее
    """
    strat_mean = df.groupby('strat')['value'].mean()
    return (strat_mean * weights).sum()


def calc_strat_var(df, weights):
    """Считает стратифицированную дисперсию.

    :param df (pd.DataFrame): датафрейм с целевой метрикой и данными для стратификации
    :param weights (pd.Series): маппинг {название страты: вес страты в популяции}
    
    :return strat_var (float): стратифицированное среднее
    """
    strat_var = df.groupby('strat')['value'].var(ddof=1)
    return (strat_var * weights).sum() # + ((1-weights) * strat_var).sum() / len(df)


def check_strat(a, b, weights):
    """Возвращает pvalue теста Стьюдента для стратифицированного среднего.

    :param a (pd.DataFrame): данные пользователей контрольной группы
    :param b (pd.DataFrame): данные пользователей экспериментальной группы
    :param weights (pd.Series): маппинг {название страты: вес страты в популяции}
    
    :return pvalue (float): pvalue
    """
    a_strat_mean = calc_strat_mean(a, weights)
    b_strat_mean = calc_strat_mean(b, weights)
    a_strat_var = calc_strat_var(a, weights)
    b_strat_var = calc_strat_var(b, weights)
    delta = b_strat_mean - a_strat_mean
    std = (a_strat_var / len(a) + b_strat_var / len(b)) ** 0.5
    t = delta / std
    df = (
        (a_strat_var / len(a) + b_strat_var / len(b)) ** 2
        / ((a_strat_var / len(a))**2 / (len(a)-1) + (b_strat_var / len(b))**2 / (len(b)-1))
    )
    pvalue = 2 * (1 - stats.t.cdf(np.abs(t), df=df))
    return pvalue


def check_ttest(a, b, weights):
    return stats.ttest_ind(a['value'], b['value']).pvalue


def get_strat_data(df: pd.DataFrame, weights: pd.Series, sample_size: int):
    strat2size = {k: int(round(v * sample_size)) for k, v in weights.to_dict().items()}
    a_dfs = []
    b_dfs = []
    for strat, df_strat in df.groupby('strat'):
        size = strat2size[strat]
        a_index, b_index = np.random.choice(
            np.arange(len(df_strat)),
            (2, size),
            False,
        )
        a_dfs.append(df_strat.iloc[a_index])
        b_dfs.append(df_strat.iloc[b_index])
    a = pd.concat(a_dfs)
    b = pd.concat(b_dfs)
    return a, b


def get_random_data(df: pd.DataFrame, weights: dict, sample_size: int):
    a_index, b_index = np.random.choice(
        np.arange(len(df)),
        (2, sample_size),
        False,
    )
    a = df.iloc[a_index]
    b = df.iloc[b_index]
    return a, b

Сгенерируем исторические данные

In [ ]:
history_size = 100_000
df_history = pd.DataFrame({
    'p1': np.random.binomial(1, 0.6, history_size),
    'p2': np.random.uniform(18, 100, history_size).astype(int),
    'value': np.random.normal(8000, 1000, history_size)
})
df_history['value'] += 1000 * (df_history['p1'] == 1)
df_history['value'] -= 30 * (df_history['p2'] - 35).abs()
df_history['value'] = df_history['value'].astype(int)

Определим страту, посчитаем веса

In [ ]:
df = df_history.copy()
df['strat'] = df['p1']
weights = df['strat'].value_counts(normalize=True)

Синтетические АА тесты

In [ ]:
n_iter = 3000
sample_size = 1000
dict_pvalues = defaultdict(list)
for _ in tqdm(range(n_iter)):
    a, b = get_random_data(df, weights, sample_size)
    dict_pvalues['srs check_ttest'].append(check_ttest(a, b, weights))
    dict_pvalues['srs check_strat'].append(check_strat(a, b, weights))
    a, b = get_strat_data(df, weights, sample_size)
    dict_pvalues['strat check_ttest'].append(check_ttest(a, b, weights))
    dict_pvalues['strat check_strat'].append(check_strat(a, b, weights))
plot_pvalue_distribution_power(dict_pvalues, alpha=0.05)

Синтетические АB тесты

In [ ]:
n_iter = 1000
sample_size = 1000
effect = 120
dict_pvalues = defaultdict(list)
for _ in tqdm(range(n_iter)):
    a, b = get_random_data(df, weights, sample_size)
    b = b.copy()
    b['value'] += effect
    dict_pvalues['srs check_ttest'].append(check_ttest(a, b, weights))
    dict_pvalues['srs check_strat'].append(check_strat(a, b, weights))
    a, b = get_strat_data(df, weights, sample_size)
    b = b.copy()
    b['value'] += effect
    dict_pvalues['strat check_ttest'].append(check_ttest(a, b, weights))
    dict_pvalues['strat check_strat'].append(check_strat(a, b, weights))
plot_pvalue_distribution_power(dict_pvalues, alpha=0.05)

#### Повышаем чувствительность
Усложним разбиение на страты

In [ ]:
df.groupby('p2')['value'].mean().plot();

In [ ]:
df = df_history.copy()
df['strat'] = (
    df['p1'].astype(str)
    + (df['p2'] < 50).astype(int).astype(str)
    + (df['p2'] > 75).astype(int).astype(str)
)
weights = df['strat'].value_counts(normalize=True)

In [ ]:
dict_pvalues = defaultdict(list)
for _ in tqdm(range(n_iter)):
    a, b = get_random_data(df, weights, sample_size)
    b = b.copy()
    b['value'] += effect
    dict_pvalues['srs check_ttest'].append(check_ttest(a, b, weights))
    dict_pvalues['srs check_strat'].append(check_strat(a, b, weights))
    a, b = get_strat_data(df, weights, sample_size)
    b = b.copy()
    b['value'] += effect
    dict_pvalues['strat check_ttest'].append(check_ttest(a, b, weights))
    dict_pvalues['strat check_strat'].append(check_strat(a, b, weights))
plot_pvalue_distribution_power(dict_pvalues, alpha=0.05)

# 4. Добавим ML

Чтобы максимально снизить дисперсию, можно использовать следующий подход:

1. Построить ML модель, которая по признакам прогнозирует значение метрики.
2. По прогнозным значениям разбить объекты на страты. С маленькими значениями прогноза в одну страту, с большими — в другую.

In [ ]:
from lightgbm import LGBMRegressor

model = LGBMRegressor()
model.fit(df[['p1', 'p2']].values, df['value'].values)
predict_test = model.predict(df[['p1', 'p2']].values)

n_strat = 100
quantiles = np.quantile(predict_test, np.linspace(0, 1 - 1 / n_strat, n_strat))
df['strat'] = [np.sum(predict >= quantiles) for predict in predict_test]

In [ ]:
weights = df['strat'].value_counts(normalize=True)

In [ ]:
dict_pvalues = defaultdict(list)
for _ in tqdm(range(n_iter)):
    a, b = get_random_data(df, weights, sample_size)
    b = b.copy()
    b['value'] += effect
    dict_pvalues['srs check_ttest'].append(check_ttest(a, b, weights))
    dict_pvalues['srs check_strat'].append(check_strat(a, b, weights))
    a, b = get_strat_data(df, weights, sample_size)
    b = b.copy()
    b['value'] += effect
    dict_pvalues['strat check_ttest'].append(check_ttest(a, b, weights))
    dict_pvalues['strat check_strat'].append(check_strat(a, b, weights))
plot_pvalue_distribution_power(dict_pvalues, alpha=0.05)

# 5. Два критерия

Могут ли критерии давать противоречивые результаты? Что с этим делать?

In [ ]:
df = df_history.copy()
df['strat'] = (
    df['p1'].astype(str)
    + (df['p2'] < 50).astype(int).astype(str)
    + (df['p2'] > 75).astype(int).astype(str)
)
weights = df['strat'].value_counts(normalize=True)

In [ ]:
dict_pvalues = defaultdict(list)
for _ in tqdm(range(10000)):
    a, b = get_random_data(df, weights, sample_size)
    dict_pvalues['AA check_ttest'].append(check_ttest(a, b, weights))
    dict_pvalues['AA check_strat'].append(check_strat(a, b, weights))
    b = b.copy()
    b['value'] += effect
    dict_pvalues['AB check_ttest'].append(check_ttest(a, b, weights))
    dict_pvalues['AB check_strat'].append(check_strat(a, b, weights))
plot_pvalue_distribution_power(dict_pvalues, alpha=0.05)

In [ ]:
df_pvalue = pd.DataFrame(dict_pvalues)

In [ ]:
alpha = 0.05
(df_pvalue < alpha).mean().round(3)

In [ ]:
(
    (df_pvalue[['AB check_ttest', 'AB check_strat']] < alpha).astype(int)
    .groupby(['AB check_ttest', 'AB check_strat'])[['AB check_strat']].count()
)

In [ ]:
df_pvalue['AA min(pvalue)'] = df_pvalue[['AA check_ttest', 'AA check_strat']].min(axis=1)
df_pvalue['AB min(pvalue)'] = df_pvalue[['AB check_ttest', 'AB check_strat']].min(axis=1)
(df_pvalue < alpha).mean()

In [ ]:
(df_pvalue <= 0.079).mean()